In [1]:
import re
from collections import Counter
from datetime import datetime
from pathlib import Path
from pprint import pprint
from typing import Optional

In [2]:
# Допустимые буквы РФ-номера (кириллица)
RF_LETTERS = 'АВЕКМНОРСТУХ'

# Латиница кириллица (путаница на вводе)
LATIN_TO_CYR = str.maketrans({
    'A': 'А', 'B': 'В', 'E': 'Е', 'K': 'К', 'M': 'М', 'H': 'Н',
    'O': 'О', 'P': 'Р', 'C': 'С', 'T': 'Т', 'Y': 'У', 'X': 'Х',
})

CAR_ID_RE = re.compile(
    rf'^([{RF_LETTERS}])(\d{{3}})([{RF_LETTERS}]{{2}})(\d{{2,3}})$'
)


def check_car_id(car_id: str) -> str:
    if not isinstance(car_id, str) or not car_id.strip():
        return 'Номер не валиден.'

    normalized = car_id.strip().upper().translate(LATIN_TO_CYR)
    match = CAR_ID_RE.fullmatch(normalized)
    if not match:
        return 'Номер не валиден.'

    letter1, digits, letters2, region = match.groups()
    plate = f'{letter1}{digits}{letters2}'
    return f'Номер {plate} валиден. Регион: {region}.'

In [3]:
print(check_car_id('А222ВС96'))
print(check_car_id('А123ВЕ777'))
print(check_car_id('АБ22ВВ193'))

Номер А222ВС валиден. Регион: 96.
Номер А123ВЕ валиден. Регион: 777.
Номер не валиден.


In [4]:
cases_1 = [
    'A222BC96',
    'А222ВС9',
    'А222ВС9699',
    'Я222ВС96',
    '',      
    '   ',
    'а222вс96',
]

for case in cases_1:
    print(repr(case), '-', check_car_id(case))

'A222BC96' - Номер А222ВС валиден. Регион: 96.
'А222ВС9' - Номер не валиден.
'А222ВС9699' - Номер не валиден.
'Я222ВС96' - Номер не валиден.
'' - Номер не валиден.
'   ' - Номер не валиден.
'а222вс96' - Номер А222ВС валиден. Регион: 96.


In [5]:
LOG_LINE_RE = re.compile(
    r'^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+'
    r'(?P<level>[A-Z]+)'
    r'(?:\s+(?P<body>.*))?$'
)

FIELD_RE = re.compile(
    r'(?i)\b(?:'
    r'user\s*=\s*(?P<user>\S+)|'
    r'action\s*=\s*(?P<action>\S+)|'
    r'ip\s*=\s*(?P<ip>\S+)|'
    r'card\s*=\s*(?P<card>\S+)'
    r')'
)

MSG_RE = re.compile(
    r'(?i)\bmsg\s*=\s*(?:"(?P<q>[^"]*)"|(?P<u>.+?))'
    r'(?=\s+\w+\s*=|$)'
)

EMAIL_RE = re.compile(
    r'^[A-Za-z0-9]+(?:[._%+-][A-Za-z0-9]+)*@'
    r'[A-Za-z0-9]+(?:[.-][A-Za-z0-9]+)*\.[A-Za-z]{1,}$'
)

_OCTET = r'(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]\d|\d)'
IPV4_RE = re.compile(rf'^(?:{_OCTET}\.){{3}}{_OCTET}$')

CARD_RE = re.compile(r'^(\d{4})-(\d{4})-(\d{4})-(\d{4})$')

DUP_WORDS_RE = re.compile(r'\b(\w+)(?:\s+\1\b)+', flags=re.UNICODE)


def _dedupe_words(text: str) -> str:
    prev = None
    while prev != text:
        prev = text
        text = DUP_WORDS_RE.sub(r'\1', text)
    return text


def _extract_fields(body: str) -> dict:
    fields = {'user': None, 'action': None, 'ip': None, 'card': None, 'message': None}
    if not body:
        return fields

    for m in FIELD_RE.finditer(body):
        for key, value in m.groupdict().items():
            if value is not None:
                fields[key] = value

    msg_m = MSG_RE.search(body)
    if msg_m:
        raw_msg = msg_m.group('q') if msg_m.group('q') is not None else msg_m.group('u')
        fields['message'] = _dedupe_words(raw_msg.strip())

    return fields


def parse_logs(text: str) -> list[dict]:
    if not isinstance(text, str) or not text.strip():
        return []

    results: list[dict] = []

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        m = LOG_LINE_RE.fullmatch(line)
        if not m:
            continue

        ts = m.group('ts')
        try:
            datetime.strptime(ts, '%Y-%m-%d %H:%M:%S')
        except ValueError:
            continue

        level = m.group('level')
        body = m.group('body') or ''

        fields = _extract_fields(body)
        user = fields['user']
        action = fields['action']
        ip = fields['ip']
        card_raw = fields['card']
        message = fields['message']

        errors: list[str] = []

        if user is None:
            errors.append('missing_user')
        elif not EMAIL_RE.fullmatch(user):
            errors.append('invalid_email')

        if action is None:
            errors.append('missing_action')

        if ip is None:
            errors.append('missing_ip')
        elif not IPV4_RE.fullmatch(ip):
            errors.append('invalid_ip')

        card: Optional[str] = None
        if card_raw is not None:
            card_match = CARD_RE.fullmatch(card_raw)
            if card_match:
                card = f'****-****-****-{card_match.group(4)}'
            elif '-' in card_raw:
                errors.append('invalid_card')

        record = {
            'ts': ts,
            'level': level,
            'user': user,
            'action': action,
            'ip': ip,
            'card': card,
            'errors': errors,
        }
        if message:
            record['message'] = message

        results.append(record)

    return results

In [6]:
LOGS_PATH = Path('logs_hw4.txt')
logs_text = LOGS_PATH.read_text(encoding='utf-8')

parsed = parse_logs(logs_text)
print('Первые 5 записей:')
pprint(parsed[:5], sort_dicts=False)


Первые 5 записей:
[{'ts': '2024-03-15 10:51:25',
  'level': 'WARN',
  'user': 'user+tag@gmail.com',
  'action': 'login',
  'ip': '203.0.113.42',
  'card': None,
  'errors': []},
 {'ts': '2024-03-15 11:46:57',
  'level': 'DEBUG',
  'user': 'petr.sidorov@yandex.ru',
  'action': 'login',
  'ip': '8.8.8.8',
  'card': None,
  'errors': [],
  'message': 'retry please'},
 {'ts': '2024-03-15 09:36:54',
  'level': 'WARN',
  'user': 'petr.sidorov@yandex.ru',
  'action': 'retry',
  'ip': '999.1.1.1',
  'card': None,
  'errors': ['invalid_ip']},
 {'ts': '2024-03-15 09:07:09',
  'level': 'ERROR',
  'user': 'user@domain.',
  'action': 'delete',
  'ip': '0.0.0.0',
  'card': None,
  'errors': ['invalid_email']},
 {'ts': '2024-03-15 10:28:11',
  'level': 'ERROR',
  'user': '@onlydomain.com',
  'action': 'pay',
  'ip': '300.300.300.300',
  'card': None,
  'errors': ['invalid_email', 'invalid_ip', 'invalid_card']}]
